In this, first of three, parts of my churn dataset portfolio project, I will demonstrate my skill in Python data creation and manipulation in order to create a relational dataset that will then be analyzed in subsequent parts.
The context: This is a dating application, with an adult userbase in Europe. We will create three datasets. The first will be about the *users*, the second will be about the *subscriptions*, and the third will show the users' *activity*.

To get started, we will create a dataset for the user database. The dataset contains basic relevant demographic variables: gender, age, sexual orientation, location, ethnicity, and work status. We will also simulate that some users freshly joined in 2025, and some churned during the year. This dataframe contains the info for all users, including inactive ones, at the end of the observation period.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

n_users = 100000
n_new = 25000
n_existing = n_users - n_new
df = pd.DataFrame()
#Assigning an ID for each user, which will uniquely identify them across datasets
df["user_id"] = np.arange(1, n_users + 1)

# date of joining app before 2025
past_dates = pd.date_range(start="2023-01-01", end="2024-12-31").to_series().values
existing_join_dates = np.random.choice(past_dates, size=n_existing)

# 25,000 users join during 2025
year_2025_dates = pd.date_range(start="2025-01-01", end="2025-12-31").to_series().values
new_join_dates = np.random.choice(year_2025_dates, size=n_new)

df["join_date"] = np.concatenate([existing_join_dates, new_join_dates])
df["join_date"] = pd.to_datetime(df["join_date"]).dt.normalize()

# Let's say 20% of users churn at some point in 2025. We will pick a date between either their date of joining (if in 2025) or the start of the year (if they joined earlier) and the end of year, and determine it as the date when they deleted their account.

df["is_churned"] = np.random.binomial(1, 0.20, size=n_users)

def assign_churn_date(rw):
    if rw["is_churned"] == 1:
        start_bound = max(rw["join_date"], pd.Timestamp("2025-01-01"))
        end_bound = pd.Timestamp("2025-12-31")
        if start_bound >= end_bound:
            return end_bound
        delta = (end_bound - start_bound).days
        return start_bound + pd.Timedelta(days=np.random.randint(0, delta + 1))
    return pd.NaT

df["churn_date"] = df.apply(assign_churn_date, axis=1)

#Creating an age distribution between 18 and 65, skewed more towards the young
shape, scale = 2.0, 6.0
df["age"] = np.random.gamma(shape, scale, n_users) + 18
df["age"] = np.clip(df["age"], 18, 65).astype(int)

#Creating genders, favouring male users, as it typical with dating apps
df["gender"] = np.random.choice(
    ['Male', 'Female', 'Non-binary'],
    size=n_users,
    p=[0.60, 0.35, 0.05]
)

#Creating sexual preference, with conditional reasoning: both male and female users have 80/15/5 ratios of preferring the opposite gender/preferring one's own gender/preferring both, while non-binary people have a random assignment. Finally, we also add a sexual orientation, depending on the gender and the preferred gender.

pref_weights = {
    'Male': [0.80, 0.15, 0.05],
    'Female': [0.15, 0.80, 0.05],
    'Non-binary': [1/3, 1/3, 1/3]
}

options = ['Prefers women', 'Prefers men', 'No preference']

df["sexual preference"] = df.groupby('gender')['gender'].transform(
    lambda x: np.random.choice(options, size=len(x), p=pref_weights.get(x.name))
)

conditions = [
    ((df["gender"] == "Male") & (df["sexual preference"] == "Prefers women")) | ((df["gender"] == "Female") & (df["sexual preference"] == "Prefers men")),
    ((df["gender"] == "Male") & (df["sexual preference"] == "Prefers men")) | ((df["gender"] == "Female") & (df["sexual preference"] == "Prefers women")),
    (df["sexual preference"] == "No preference"),
    (df["gender"] == "Non-binary")
]

choices = [
    "Heterosexual",
    "Homosexual",
    "Bisexual",
    "Non-applicable"
]

df["sexual orientation"] = np.select(conditions, choices, default="Other")

#Creating locations, limited to a couple per country for the sake of simplicity. The simulated data will take a country with assigned probabilities and then take a random city from that country.
locations_map = {
    'United Kingdom': ['London', 'Manchester', 'Birmingham', 'Edinburgh'],
    'Germany': ['Berlin', 'Munich', 'Hamburg', 'Cologne'],
    'France': ['Paris', 'Lyon', 'Marseille', 'Toulouse'],
    'Spain': ['Madrid', 'Barcelona', 'Valencia', 'Seville'],
    'Italy': ['Rome', 'Milan', 'Naples', 'Turin'],
    'Netherlands': ['Amsterdam', 'Rotterdam', 'The Hague', 'Utrecht']
}

countries = list(locations_map.keys())
country_probs = [0.25, 0.20, 0.20, 0.15, 0.15, 0.05]
df["country"] = np.random.choice(countries, size=n_users, p=country_probs)

df["city"] = [np.random.choice(locations_map[c]) for c in df["country"]]

df["ethnicity"] = np.random.choice(
    ['White/Caucasian', 'Asian', 'Hispanic/Latino', 'Black/African Descent', 'Mixed/Other'],
    size=n_users,
    p=[0.65, 0.10, 0.10, 0.08, 0.07]
)

df["work_status"] = np.random.choice(
    ['Employed', 'Student', 'Unemployed'],
    size=n_users,
    p=[0.70, 0.20, 0.10]
)

df_users = df.copy()

print(df_users.head())
print("\n")
print(df_users.info())

 Thus, we created a simple, randomized dataset of our user base. The next step is to create the user churn. Focusing on one year (2025), we will have the base-state of each user, and then the date on which they switched memberships between free, plus, and gold variants of the app.

In [ ]:
# Setting n of users and user_ids
n_users = 100000
user_ids = np.arange(1, n_users + 1)

activity_records = []

# Log "account_created" for users joining in 2025
new_users = df[df['join_date'].dt.year == 2025]

for _, row in new_users.iterrows():
    activity_records.append({
        'user_id': row['user_id'],
        'date': row['join_date'],
        'action': 'account_created'
    })
    # They start as 'free' upon creation
    activity_records.append({
        'user_id': row['user_id'],
        'date': row['join_date'],
        'action': 'free'
    })

# Log "account_deleted" for churned users
churned_users = df[df['is_churned'] == 1]
for _, row in churned_users.iterrows():
    activity_records.append({
        'user_id': row['user_id'],
        'date': row['churn_date'],
        'action': 'account_deleted'
    })

# Establishing the base-state at the start of the year (60% Free, 30% Plus, 10% Gold)
# New 2025 users are always 'free'; only existing users get a random initial tier
initial_states = np.where(
    df['join_date'].dt.year == 2025,
    'free',
    np.random.choice(['free', 'plus', 'gold'], size=n_users, p=[0.60, 0.30, 0.10])
)

# FIX: Log the initial state of existing users on 2025-01-01
# (new users already have their 'free' state logged above on their join date)
existing_users = df[df['join_date'].dt.year < 2025]
for _, row in existing_users.iterrows():
    activity_records.append({
        'user_id': row['user_id'],
        'date': pd.Timestamp('2025-01-01'),
        'action': initial_states[row['user_id'] - 1]  # user_ids are 1-indexed
    })

# 70% of users don't change their tier, 20% switch once, 8% twice, 2% three times
n_switches = np.random.choice(
    [0, 1, 2, 3],
    size=n_users,
    p=[0.70, 0.20, 0.08, 0.02]
)

# Filter down to only users who actually make a switch
switching_user_indices = np.where(n_switches > 0)[0]

# FIX: Pre-index df by user_id to avoid a full scan on every loop iteration
df_indexed = df.set_index('user_id')

for idx in switching_user_indices:
    u_id = user_ids[idx]

    # Get user lifespan bounds for 2025
    user_info = df_indexed.loc[u_id]
    valid_start = max(user_info['join_date'], pd.Timestamp('2025-01-01'))
    valid_end = user_info['churn_date'] if pd.notnull(user_info['churn_date']) else pd.Timestamp('2025-12-31')

    start_ts = valid_start.value // 10**9
    end_ts = valid_end.value // 10**9

    if start_ts < end_ts:
        switches_count = n_switches[idx]
        current_state = initial_states[idx]

        random_timestamps = np.random.randint(start_ts, end_ts, size=switches_count)
        random_timestamps.sort()
        switch_dates = pd.to_datetime(random_timestamps, unit='s').normalize()

        for date in switch_dates:
            possible_states = ['free', 'plus', 'gold']
            possible_states.remove(current_state)
            next_state = np.random.choice(possible_states)

            activity_records.append({
                'user_id': u_id,
                'date': date,
                'action': next_state
            })
            current_state = next_state

df_subscription = pd.DataFrame(activity_records).sort_values(by=['date', 'user_id']).reset_index(drop=True)

print(df_subscription.head(10))
print("\n")
print(df_subscription.info())
print("\n")
print("Action counts:")
print(df_subscription['action'].value_counts())

In doing that, we created our actual churn. Now that we know who are users are and when they switched memberships, we want to also have the data on what they did on the app throughout the year. This dataset will contain the number of likes, the number of dislikes, the number of matches, the number of messages received, and the number of messages sent.
In order to make the data more realistic, I assigned free users to have up to 20 likes/dislikes allowed per day, plus users to have up to 100, and unlimited liking/disliking for the gold tier. This limit had to be dynamically updated throughout the dataset, based on the churn from the previous dataset.

For the sake of realism and manageability within my offline system, we will assume that not every user had activity every day. Also, new users will only have activity after they join. Finally, we will introduce some variation into the dataset, so that not everything is random - we will add the user factors to the dataset to do this, but then we will remove them, because they are not what this dataset represents.

In [ ]:
# As previously: users, ids
n_users = 100000
user_ids = np.arange(1, n_users + 1)

# Ensure df_subscription is clean and formatted for the tier timeline
df_subscription = df_subscription.copy()

tier_actions = ['free', 'plus', 'gold']
df_subscription['tier'] = df_subscription['action'].where(
    df_subscription['action'].isin(tier_actions)
).groupby(df_subscription['user_id']).ffill()

# Create the tier timeline for merge_asof
tier_timeline = df_subscription[['user_id', 'date', 'tier']].sort_values(['date', 'user_id'])

# Calculate active days based on their 'starting' tier for logic simplicity
activity_lambdas = {'free': 30, 'plus': 80, 'gold': 150}
n_active_days = np.random.poisson(pd.Series(initial_states).map(activity_lambdas).fillna(30))
n_active_days = np.clip(n_active_days, 1, 365)

# Create the backbone of the dataset
active_user_ids = np.repeat(df['user_id'].values, n_active_days)
total_rows = len(active_user_ids)

#add seasonality to the dataset, so that we have more activity on weekends, during the summer, and around Valentine's Day
all_2025_dates = pd.date_range('2025-01-01', '2025-12-31')
date_weights = np.ones(len(all_2025_dates))

for i, d in enumerate(all_2025_dates):
    if d.weekday() >= 5:
        date_weights[i] *= 2.0
    if d.month == 2 and 10 <= d.day <= 15:
        date_weights[i] *= 3.0
    if 5 < d.month < 9:
        date_weights[i] *= 1.5

date_probs = date_weights / date_weights.sum()

# now we generate random dates using the seasonal weights
random_dates = np.random.choice(all_2025_dates, size=total_rows, p=date_probs)

df_daily = pd.DataFrame({
    'user_id': active_user_ids,
    'date': random_dates
}).sort_values(['date','user_id'])

## Pull demographic bounds from the main users dataset
user_bounds = df[['user_id', 'join_date', 'churn_date', 'age', 'gender']].copy()

# Force them into datetime objects just to be absolutely safe
user_bounds['join_date'] = pd.to_datetime(user_bounds['join_date'])
user_bounds['churn_date'] = pd.to_datetime(user_bounds['churn_date'])

# Use Pandas-native .clip() instead of np.maximum to avoid TypeError
user_bounds['valid_start'] = user_bounds['join_date'].clip(lower=pd.Timestamp('2025-01-01'))
user_bounds['valid_end'] = user_bounds['churn_date'].fillna(pd.Timestamp('2025-12-31'))

# Merge bounds into the daily activity log
df_daily = df_daily.merge(user_bounds, on='user_id')

# Filter out dates where the user hadn't joined yet or had already deleted their account
df_daily = df_daily[(df_daily['date'] >= df_daily['valid_start']) &
                    (df_daily['date'] <= df_daily['valid_end'])]
df_daily = df_daily.sort_values(['date', 'user_id'])

# Vectorized join to map states to dates (tier at the moment of activity)
df_daily = pd.merge_asof(
    df_daily,
    tier_timeline,
    on='date',
    by='user_id',
    direction='backward'
)

# Drop rows where tier is missing (activity slipped in before their first logged tier)
df_daily = df_daily.dropna(subset=['tier'])

# Initialize columns for likes and dislikes
df_daily['likes'] = 0
df_daily['dislikes'] = 0

# Masks for different tier behaviors
is_free = df_daily['tier'] == 'free'
is_plus = df_daily['tier'] == 'plus'
is_gold = df_daily['tier'] == 'gold'

# Free: Binomial around 15, max 20
df_daily.loc[is_free, 'likes'] = np.random.binomial(20, 0.4, size=is_free.sum())
df_daily.loc[is_free, 'dislikes'] = np.random.binomial(20, 0.4, size=is_free.sum())

# Plus: Binomial around 60, max 100
df_daily.loc[is_plus, 'likes'] = np.random.binomial(100, 0.5, size=is_plus.sum())
df_daily.loc[is_plus, 'dislikes'] = np.random.binomial(100, 0.3, size=is_plus.sum())

# Gold: Log-normal for gold members because we'd expect there to be some heavy users
gold_swipes = np.random.lognormal(4.2, 0.7, size=is_gold.sum())
df_daily.loc[is_gold, 'likes'] = (gold_swipes * 0.6).astype(int)
df_daily.loc[is_gold, 'dislikes'] = (gold_swipes * 0.4).astype(int)


# Increase men's likes, increase women's dislikes
male_mask = df_daily['gender'] == 'Male'
female_mask = df_daily['gender'] == 'Female'

df_daily.loc[male_mask, 'likes'] = (df_daily.loc[male_mask, 'likes'] * 1.5).astype(int)
df_daily.loc[female_mask, 'dislikes'] = (df_daily.loc[female_mask, 'dislikes'] * 1.5).astype(int)

# Apply a multiplier to younger users (< 30 years old)
age_multiplier = np.where(df_daily['age'] < 30, 1.3, 1.0)

df_daily['likes'] = (df_daily['likes'] * age_multiplier).astype(int)
df_daily['dislikes'] = (df_daily['dislikes'] * age_multiplier).astype(int)

# Higher tiers get better "conversion" rates
match_rates = {'free': 0.04, 'plus': 0.05, 'gold': 0.06, 'account_deleted':0}
msg_rates = {'free': 0.5, 'plus': 1.2, 'gold': 2, 'account_deleted':0}

# Calculate matches based on updated likes and tier-based probability
probs = df_daily['tier'].map(match_rates)
df_daily['matches'] = np.random.binomial(df_daily['likes'].astype(int), probs)

# Calculate messages based on matches
msg_lambdas = df_daily['matches'] * df_daily['tier'].map(msg_rates)
shared_component = np.random.poisson(msg_lambdas * 0.25)
df_daily['messages_sent'] = shared_component + np.random.poisson(msg_lambdas * 0.75)
df_daily['messages_received'] = shared_component + np.random.poisson(msg_lambdas * 0.75)

# Finally, we remove the data which we already have in the main user df
columns_to_drop = ['tier', 'valid_start', 'valid_end', 'join_date', 'churn_date', 'age', 'gender']
df_daily = df_daily.drop(columns=[col for col in columns_to_drop if col in df_daily.columns])
df_daily = df_daily.sort_values(['date', 'user_id']).reset_index(drop=True)

print(f"Total activity rows generated (post-filtering): {len(df_daily):,}")
print(df_daily.head())

Now we have our three datasets. The next thing we are going to do is explore our data. Let's see who our users are, how much do they use the app, when, etc. For this part of the project, we are going to stick to considering them separately.

First, we will look at the demographics, extracting simple statistics like means and ranges, and comparing groups across genders, orientations, and locations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style for the plots
sns.set_theme(style ="white")

# 1. Basic Statistical Summaries
print("--- User Demographics Summary ---")
print(f"Age Range: {df_users['age'].min()} to {df_users['age'].max()} years")
print(f"Average Age: {df_users['age'].mean():.1f} years\n")

print("--- Average Age by Gender ---")
print(df_users.groupby('gender')['age'].mean().round(1).to_string())
print("\n")

print("--- User Count by Sexual Orientation ---")
print(df_users['sexual orientation'].value_counts().to_string())
print("\n")

print("--- User Count by Country ---")
print(df_users['country'].value_counts().to_string())

# Visualizing the Demographics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('User Demographics Exploration', fontsize=18)

# Top-Left: Age Distribution
sns.histplot(data=df_users, x='age', bins=20, kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Age Distribution')
axes[0, 0].set_xlabel('Age')

# Top-Right: Gender and Sexual Orientation
sns.countplot(data=df_users, x='gender', hue='sexual orientation', ax=axes[0, 1])
axes[0, 1].set_title('Sexual Orientation by Gender')
axes[0, 1].set_xlabel('Gender')
axes[0, 1].legend(title='Orientation')

# Bottom-Left: Country Distribution
sns.countplot(data=df_users, y='country', ax=axes[1, 0], order=df_users['country'].value_counts().index)
axes[1, 0].set_title('Users per Country')
axes[1, 0].set_xlabel('Count')
axes[1, 0].set_ylabel('')

# Bottom-Right: Work Status by Gender
sns.countplot(data=df_users, x='work_status', hue='gender', ax=axes[1, 1], palette='pastel')
axes[1, 1].set_title('Work Status by Gender')
axes[1, 1].set_xlabel('Work Status')
axes[1, 1].legend(title='Gender')

plt.tight_layout()
plt.show()

Of course, since it's simulated data, we already know that it's mostly young people, mostly heterosexual, and employed. However, similar stats would be very valuable for a real-world dataset. Next, we will look at the evolution of our subscription tiers across 2025. Since our `df_subscription` dataset only logs the *events* (when someone switched), we will calculate the daily net change of users in each tier to build a continuous timeline of our subscriber base.

In [ ]:
# 1. Map actions to valid subscription states
tier_map = {
    'free': 'free',
    'plus': 'plus',
    'gold': 'gold',
    'account_deleted': 'deleted'
}

timeline = df_subscription[['user_id', 'date', 'action']].copy()
timeline['state'] = timeline['action'].map(tier_map)
# Drop unmapped actions (i.e. 'account_created') — no state to track
timeline = timeline.dropna(subset=['state'])

# 2. Recreate the initial states baseline for pre-2025 users
# This anchors existing users so we can correctly compute their first loss/gain in 2025
initial_states_df = pd.DataFrame({'user_id': df['user_id'], 'tier': initial_states})
existing_users = df[df['join_date'] < '2025-01-01']['user_id']

baseline_states = initial_states_df[initial_states_df['user_id'].isin(existing_users)].copy()
baseline_states['date'] = pd.Timestamp('2024-12-31')
baseline_states['state'] = baseline_states['tier']
baseline_states = baseline_states[['user_id', 'date', 'state']]

# 3. Combine baseline and events, then track state changes
full_timeline = pd.concat([baseline_states, timeline]).sort_values(['user_id', 'date'])
full_timeline = full_timeline.drop_duplicates(subset=['user_id', 'date'], keep='last')

# Shift to find what tier each user was in previously
full_timeline['prev_state'] = full_timeline.groupby('user_id')['state'].shift(1)

# 4. Calculate daily gains and losses (2025 events only)
events_2025 = full_timeline[full_timeline['date'] >= '2025-01-01'].copy()

# Gain: moving INTO free, plus, or gold
is_gain = (events_2025['state'] != 'deleted') & (events_2025['prev_state'] != events_2025['state'])
gains = events_2025[is_gain].groupby(['date', 'state']).size().rename('change')

# Loss: leaving a tier (switching to a new one or deleting account)
is_loss = events_2025['prev_state'].notna() & (events_2025['state'] != events_2025['prev_state'])
losses = events_2025[is_loss].groupby(['date', 'prev_state']).size().rename('change') * -1
losses.index.names = ['date', 'state']

# 5. Combine, reindex, and calculate cumulative sum
net_changes = pd.concat([gains, losses]).groupby(['date', 'state']).sum().unstack(fill_value=0)

# Ensure all days of the year are present
all_days = pd.date_range('2025-01-01', '2025-12-31')
net_changes = net_changes.reindex(all_days, fill_value=0)

# Seed Jan 1st with the existing user population baseline
initial_counts = baseline_states['state'].value_counts()
for state, count in initial_counts.items():
    if state in net_changes.columns:
        net_changes.loc['2025-01-01', state] += count

daily_tier_pop = net_changes.cumsum()

# 6. Plot
plt.figure(figsize=(12, 6))
colors = {'free': 'gray', 'plus': 'dodgerblue', 'gold': 'gold'}

for tier in ['free', 'plus', 'gold']:
    if tier in daily_tier_pop.columns:
        plt.plot(daily_tier_pop.index, daily_tier_pop[tier], label=tier.capitalize(), color=colors[tier], linewidth=2)

plt.title('Subscription Tier Evolution (2025)', fontsize=16)
plt.xlabel('Date')
plt.ylabel('Number of Users')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Here, we can see that over the year, there was a growth in popularity of the gold tier, and a smaller growth in the popularity of the plus tier. There was a reduction in free tier members, which is necessary since we kept the total number of users constant. In a real dataset, of course, these three could be relatively independent due to people only starting to use the app or ending their usage.

Next, we can have a look at how many users joined and how many left over the timespan

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# Calculate Monthly Inflow (New Joins)
new_joins_monthly = df[df['join_date'].dt.year == 2025].resample('ME', on='join_date').size()

# Calculate Monthly Outflow (Churns)
churn_events = events_2025[events_2025['action'] == 'account_deleted']

if not churn_events.empty:
    churns_monthly = churn_events.resample('ME', on='date').size()
else:
    churns_monthly = pd.Series(0, index=new_joins_monthly.index)

churns_monthly = churns_monthly.reindex(new_joins_monthly.index, fill_value=0)

# --- THE FIX: CENTER THE DATES ---
months_centered = new_joins_monthly.index.to_period('M').to_timestamp() + pd.Timedelta(days=14)

# --- Single Figure with Twin Axes ---
plt.style.use('dark_background')
fig, ax1 = plt.subplots(figsize=(14, 7), facecolor='#2c2c2c')
ax1.set_facecolor('#242b30')

# Define a "muted white" color for the UI elements (White with 0.7 alpha)
muted_white = (1, 1, 1, 0.7)

# LEFT AXIS: Bars (Inflow / Outflow)
bar_width = 8
bars1 = ax1.bar(months_centered - pd.Timedelta(days=5), new_joins_monthly, width=bar_width,
                label='New Registrations', color='#2ecc71', alpha=0.8)
bars2 = ax1.bar(months_centered + pd.Timedelta(days=5), churns_monthly, width=bar_width,
                label='Account Deletions (Churn)', color='#e74c3c', alpha=0.8)

ax1.set_ylabel('Monthly Users (Inflow / Outflow)', color='white', alpha=0.7)
# FIXED: Removed 'alpha' and used the RGBA color tuple instead
ax1.tick_params(axis='y', colors=muted_white)
ax1.grid(axis='y', linestyle='--', alpha=0.2)

# X-axis formatting: Force strictly to 2025
ax1.set_xlim([pd.Timestamp('2025-01-01'), pd.Timestamp('2025-12-31')])
ax1.xaxis.set_major_locator(plt.FixedLocator(mdates.date2num(months_centered)))
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax1.tick_params(axis='x', colors=muted_white)

# RIGHT AXIS: Line (Total Active Users)
ax2 = ax1.twinx()
total_active = daily_tier_pop.sum(axis=1)
total_active_2025 = total_active['2025-01-01':'2025-12-31']

line, = ax2.plot(total_active_2025.index, total_active_2025, color='white', linewidth=2.5,
                 label='Total Active Users', zorder=5, alpha=0.8)

ax2.fill_between(total_active_2025.index, total_active_2025, color='skyblue', alpha=0.1, zorder=4)
ax2.set_ylim(0, total_active_2025.max() * 1.15)
ax2.set_ylabel('Total Active Users (Daily)', color='white', alpha=0.7)
ax2.tick_params(axis='y', colors=muted_white)

# Combined legend
handles = [bars1, bars2, line]
labels = [h.get_label() for h in handles]
ax1.legend(handles, labels, loc='upper left', framealpha=0.1, fontsize=10)

plt.title('Active User Base & Monthly Dynamics (2025)', fontsize=16, fontweight='bold', color='white', alpha=0.8)
plt.tight_layout()
plt.show()



Finally, let's see how the users actually engaged with the app. We will aggregate the daily activity log on a weekly basis to smooth out daily variance, and split the interactions into three clear funnels: Swiping (Likes/Dislikes), Matching, and Messaging.

In [ ]:
# Select only the columns we actually want to plot before resampling.
# Without this, numeric_only=True would also sum user_id, which is meaningless.
activity_cols = ['date', 'likes', 'dislikes', 'matches', 'messages_sent', 'messages_received']
daily_activity = df_daily[activity_cols].set_index('date').resample('D').sum()

# Apply a 3-day rolling average to smooth noise without losing granularity.
# min_periods=1 ensures we still get values at the edges of the year
# rather than NaN for the first two days.
rolling_activity = daily_activity.rolling(window=3, min_periods=1).mean()

# Set up a 3-subplot figure sharing the X-axis
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
fig.suptitle('App Activity Logs — 3-Day Rolling Average (2025)', fontsize=18)

# Graph 1: Likes and Dislikes
ax1.plot(rolling_activity.index, rolling_activity['likes'], label='Likes', color='green')
ax1.plot(rolling_activity.index, rolling_activity['dislikes'], label='Dislikes', color='red')
ax1.set_title('Daily Swipes (Likes & Dislikes)')
ax1.set_ylabel('Total Count')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Graph 2: Matches
ax2.plot(rolling_activity.index, rolling_activity['matches'], label='Matches', color='purple')
ax2.set_title('Daily Matches')
ax2.set_ylabel('Total Matches')
ax2.legend()
ax2.grid(True, alpha=0.3)

# Graph 3: Messages Sent and Received
ax3.plot(rolling_activity.index, rolling_activity['messages_sent'], label='Messages Sent', color='dodgerblue', linestyle='--')
ax3.plot(rolling_activity.index, rolling_activity['messages_received'], label='Messages Received', color='darkorange', alpha=0.7)
ax3.set_title('Daily Messaging Volume')
ax3.set_xlabel('Date')
ax3.set_ylabel('Total Messages')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()